# Mini Project 3: Iris Flower Clustering with K-Means

In this notebook, I apply K-Means clustering to the Iris dataset. I scale the four numerical features, use the elbow method to inspect possible values of K, fit K-Means with `k=3`, print the cluster centers, visualize the clusters, and compare the cluster assignments with the true Iris species labels.

**Important:** K-Means is unsupervised. It does not use the true species labels to form clusters, and cluster IDs are arbitrary.

## 1. Import libraries and load the Iris dataset

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
import joblib
import pickle

# I run this notebook from the notebooks/ directory.
ROOT = Path("..")
IMAGES_DIR = ROOT / "images"
MODELS_DIR = ROOT / "models"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

iris = load_iris(as_frame=True)
X = iris.data.copy()
y = iris.target.copy()

df = X.copy()
df["species"] = y.map(dict(enumerate(iris.target_names)))
display(df.head())
print("Dataset shape:", X.shape)
print("Feature names:", list(X.columns))
print("Species:", list(iris.target_names))

## 2. Scale the features

I standardize the features so that measurements with larger numerical ranges do not dominate the distance calculations used by K-Means.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled data shape:", X_scaled.shape)
print("Feature means after scaling (approximately 0):", X_scaled.mean(axis=0).round(3))
print("Feature standard deviations (approximately 1):", X_scaled.std(axis=0).round(3))

## 3. Use the elbow method to inspect K

I fit K-Means for several values of K and plot inertia (within-cluster sum of squared distances). The elbow is a useful visual guide, not a guaranteed way to identify the only correct K.

In [ ]:
k_values = range(1, 11)
inertias = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertias, marker="o")
plt.title("Elbow Method for Iris Dataset")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.xticks(list(k_values))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "01_elbow_method.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Fit K-Means with k=3 and print cluster centers

I use `k=3` for this mini project because the Iris dataset contains three known species. The true labels are not supplied to K-Means during fitting.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

print("Cluster centers in standardized feature space:")
display(pd.DataFrame(kmeans.cluster_centers_, columns=X.columns).round(3))

# Convert centers back to the original measurement units for easier interpretation.
centers_original = scaler.inverse_transform(kmeans.cluster_centers_)
centers_df = pd.DataFrame(centers_original, columns=X.columns)
centers_df.index.name = "Cluster"
print("Cluster centers in original feature units:")
display(centers_df.round(3))

print("Inertia:", round(kmeans.inertia_, 3))
print("Silhouette score:", round(silhouette_score(X_scaled, cluster_labels), 3))

## 5. Visualize clusters with Matplotlib

I use PCA to project the four scaled features into two dimensions for plotting. The clustering itself was performed on all four scaled features, not just the two plotted components.

In [ ]:
pca_for_plot = PCA(n_components=2)
X_2d = pca_for_plot.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    X_2d[:, 0], X_2d[:, 1],
    c=cluster_labels, cmap="viridis", s=55, edgecolor="black", linewidth=0.3
)
plt.title("K-Means Clusters on Iris (PCA 2D Projection)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(*scatter.legend_elements(), title="Cluster")
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "02_kmeans_clusters_pca.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Compare predicted clusters with true species labels

The contingency table shows how samples from each real species are distributed across the clusters. The Adjusted Rand Index (ARI) compares the two groupings while accounting for arbitrary cluster numbering; 1 means identical groupings and values near 0 indicate similarity around chance.

In [ ]:
comparison = pd.crosstab(
    pd.Series(y.map(dict(enumerate(iris.target_names))), name="True species"),
    pd.Series(cluster_labels, name="Predicted cluster")
)
display(comparison)

ari = adjusted_rand_score(y, cluster_labels)
print(f"Adjusted Rand Index (ARI): {ari:.3f}")

# Visualize the true species labels in the same 2D PCA space for comparison.
plt.figure(figsize=(8, 6))
for species_id, species_name in enumerate(iris.target_names):
    mask = y.to_numpy() == species_id
    plt.scatter(
        X_2d[mask, 0], X_2d[mask, 1],
        label=species_name, s=55, edgecolor="black", linewidth=0.3
    )
plt.title("True Iris Species (PCA 2D Projection)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="True species")
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "03_true_species_pca.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Save and reload the model with joblib and pickle

I save the fitted scaler and K-Means model so I can reuse them later without training again. In a real application, I should keep the preprocessing steps together with the model.

In [ ]:
# Save the scaler and K-Means model using joblib.
joblib.dump(scaler, MODELS_DIR / "iris_scaler.joblib")
joblib.dump(kmeans, MODELS_DIR / "iris_kmeans.joblib")

# Save the fitted K-Means model using pickle as an alternative.
with open(MODELS_DIR / "iris_kmeans.pkl", "wb") as file:
    pickle.dump(kmeans, file)

# Reload and check that predictions match.
loaded_scaler = joblib.load(MODELS_DIR / "iris_scaler.joblib")
loaded_kmeans = joblib.load(MODELS_DIR / "iris_kmeans.joblib")
sample_scaled = loaded_scaler.transform(X.iloc[:5])
loaded_predictions = loaded_kmeans.predict(sample_scaled)

with open(MODELS_DIR / "iris_kmeans.pkl", "rb") as file:
    pickled_kmeans = pickle.load(file)

print("First 5 cluster predictions from reloaded joblib model:", loaded_predictions)
print("Predictions match original model:", np.array_equal(loaded_predictions, kmeans.predict(X_scaled[:5])))
print("Predictions from pickle model:", pickled_kmeans.predict(X_scaled[:5]))

## 8. My conclusion

In this mini project, I applied K-Means clustering to the scaled Iris dataset using three clusters. I printed the cluster centers in both standardized and original feature units, visualized the groups using a two-dimensional PCA projection, and compared the resulting clusters with the true species labels. I also used the elbow method to inspect different values of K and demonstrated how to save and reload a trained model using joblib and pickle.

K-Means does not learn from the species labels, so its cluster numbers do not directly correspond to the species names. The comparison metrics help me understand how closely the discovered groups align with the known labels.